<a href="https://colab.research.google.com/github/HrishiKabra/AI_Engineering_Project/blob/main/AI_Engineering_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# F1 Rule & Penalty Interpreter

**Team Members:** Hrishi Kabra & Ayush Bhatia

## Project Overview

Our project is an AI powered assistant designed to help Formula 1 fans understand the complex FIA regulations and steward decisions. Penalties and rules in Formula 1 are quite difficult to interpret and often span across long and cross referenced rulebooks. By using a RAG pipeline, our system can ingest FIA sporting regulations and Steward Decision documents to translate these legal eplanations into plain English.

Both of us are passionate Formula 1 fans and we are building this to help watchers understand these complicated rules.

## Current Status
We have a working MVP that successfully ingests the 2025 F1 regulations and decision document pdfs into a knowledge base. It routes the questions, retrieves the relevant passages and uses the OpenAI api key to help create answers. The gradio app is working and can answer questions through rag and can identify f1 documents.

Current failure - Our chunk size is set to 300 - because of this the semantic search matches the introductory headers of the steward decision docs rather than the actual reasoning below - causes llm to hallucinate the penalty justifications - loses context.

## 1) Setup


In [ ]:
# Install lightweight dependencies.
# Keep installs minimal for student reliability.
!pip -q install gradio pandas numpy pypdf openai

import os, re, json, time, math, hashlib
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

# Create folders for logs/results (safe if they already exist)
os.makedirs("runs", exist_ok=True)
os.makedirs("data", exist_ok=True)


In [ ]:
# Standard setup cell
# Clones your specific GitHub repository
!git clone --depth 1 -q https://github.com/HrishiKabra/AI_Engineering_Project.git /content/project_repo

!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git /content/main_course_repo
import sys; sys.path.append('/content/main_course_repo')
from course_utils import lab5_setup, get_text_embedding
lab5_setup()


fatal: destination path '/content/project_repo' already exists and is not an empty directory.
fatal: destination path '/content/main_course_repo' already exists and is not an empty directory.
🔧 Setting up your environment...
  → Installing core packages...
  → Setting random seed for reproducible results...
  → Checking API key...
✅ OPENAI_API_KEY already set.
  → Adding course files to path...
✅ Setup complete!
✅ lab5_setup complete — ready.


In [ ]:
from pathlib import Path
from pypdf import PdfReader

def pdf_to_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        txt = page.extract_text() or ""
        parts.append(txt)
    return "\n".join(parts)

def load_pdfs(folder: Path):
    docs = []
    for p in sorted(folder.rglob("*.pdf")):
        text = pdf_to_text(p).strip()
        if len(text) > 500:  # avoid tiny/empty extracts
            docs.append({"source": str(p), "text": text})
    return docs

## 2) Configuration


In [ ]:
@dataclass
class Config:
    # LLM
    model: str = "mock-model"
    temperature: float = 0.2
    max_tokens: int = 400
    use_mock_llm: bool = False  # ✅ set False when you wire a real API

    # RAG
    use_rag: bool = True
    chunk_size: int = 300
    chunk_overlap: int = 50
    top_k: int = 4

    # Tools / Router
    use_router: bool = True

    # Safety & trust
    guardrails_on: bool = True
    require_citations: bool = True
    low_confidence_threshold: float = 0.25  # heuristic for warnings

    # Logging / Ops
    log_path: str = "runs/interactions.jsonl"

cfg = Config()
cfg


Config(model='mock-model', temperature=0.2, max_tokens=400, use_mock_llm=False, use_rag=True, chunk_size=300, chunk_overlap=50, top_k=4, use_router=True, guardrails_on=True, require_citations=True, low_confidence_threshold=0.25, log_path='runs/interactions.jsonl')

## 3) Data (Knowledge Base)
Our data consists of official FIA Formula One Sporting Regulations and Steward Decisions from the 2025/2026 seasons.
- **Sources:** [FIA Regulations](https://www.fia.com/regulation/category/110) & [FIA Documents](https://www.fia.com/documents/)


In [ ]:
from pathlib import Path
# Point to the cloned repository folder
BASE = Path("/content/project_repo/docs")
REGS = BASE / "regulations"
DECS = BASE / "decision_docs"

reg_docs = load_pdfs(REGS)
dec_docs = load_pdfs(DECS)
docs = reg_docs + dec_docs

print("regulations:", len(reg_docs))
print("decisions:", len(dec_docs))
print("total:", len(docs))
if docs:
    print("example source:", docs[0]["source"])
    print(docs[0]["text"][:800])
else:
    print("No PDFs found. Make sure the github repo URL is correct and contains the docs folder.")


regulations: 2
decisions: 1305
total: 1307
example source: /content/project_repo/docs/regulations/fia_2025_f1_guidelines_penalty_points_overview.pdf
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start for both 
Competitor cars (mandatory) 
0 
Breach of “Covers On” Time 23.6 & 38.2a) i)     Pit lane start for both 
Competitor cars (mandatory) 
 
Receiving Mechanical Assistance 
Re-joining by use of mechanical assistance 26.4     Disqualification 0 
Use of Tyres 
Use of tyres without appropriate iden


In [ ]:
KB = []

def add_folder_to_kb(folder, prefix):
    for pdf in folder.rglob("*.pdf"):
        text = pdf_to_text(pdf)

        if len(text) < 200:
            continue

        KB.append({
            "id": f"{prefix}_{pdf.stem}",
            "title": pdf.stem,
            "text": text,
            "source": str(pdf)
        })

add_folder_to_kb(REGS, "regulation")
add_folder_to_kb(DECS, "decision")

print("documents loaded:", len(KB))
print(KB[0]["title"])
print(KB[0]["text"][:500])

documents loaded: 1364
fia_2025_f1_guidelines_penalty_points_overview
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start fo


### 3.1) Document Chunking


In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    # Simple character-based chunker (good enough to start)
    if chunk_size <= 0:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

def build_chunks(kb: List[Dict[str, Any]], cfg: Config) -> List[Dict[str, Any]]:
    chunks = []
    for doc in kb:
        for i, chunk in enumerate(chunk_text(doc["text"], cfg.chunk_size, cfg.chunk_overlap)):
            chunks.append({
                "chunk_id": f'{doc["id"]}::chunk{i}',
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": chunk,
                "source": doc.get("source", ""),
            })
    return chunks

CHUNKS = build_chunks(KB, cfg)
len(CHUNKS), CHUNKS[0]["chunk_id"]


(17249, 'regulation_fia_2025_f1_guidelines_penalty_points_overview::chunk0')

## 4) Core Components
### Logging


In [ ]:
def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def append_jsonl(path: str, record: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def stable_hash(obj: Any) -> str:
    raw = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:12]


### LLM Wrapper


In [ ]:
from openai import OpenAI
import os

client = OpenAI()

def call_llm_real_api(prompt: str, cfg: Config) -> Dict[str, Any]:
    """
    Calls the real OpenAI API.
    """
    # Default to gpt-4o-mini if 'mock-model' is still in config
    model_name = cfg.model if cfg.model != 'mock-model' else 'gpt-4o-mini'

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant tasked with answering questions about Formula 1 regulations and decisions."},
                {"role": "user", "content": prompt}
            ],
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
        )
        text = response.choices[0].message.content

        # Optional usage parsing
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    except Exception as e:
        text = f"Error calling OpenAI API: {e}"
        usage = {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None}

    return {
        "text": text,
        "usage": usage,
        "model": model_name,
        "latency_ms": 0.0,
        "cost_usd": 0.0
    }

def call_llm_mock(prompt: str, cfg: Config) -> Dict[str, Any]:
    response = (
        "MOCK ANSWER:\n"
        f"- I received your question/prompt: {prompt[:180]!r}\n"
        "- In a real project, this would be answered using the selected tool(s) and/or RAG.\n"
        "- Now providing a concise response placeholder."
    )
    return {
        "text": response,
        "usage": {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None},
        "model": cfg.model,
        "latency_ms": 0.5,
        "cost_usd": 0.0
    }

import time
def call_llm(prompt: str, cfg: Config) -> Dict[str, Any]:
    t0 = time.time()
    if getattr(cfg, 'use_mock_llm', False):
        out = call_llm_mock(prompt, cfg)
    else:
        out = call_llm_real_api(prompt, cfg)

    out['latency_ms'] = round((time.time() - t0) * 1000, 2)
    return out


### Retrieval System


In [ ]:
def keyword_score(query: str, text: str) -> float:
    q = re.findall(r"[a-zA-Z0-9']+", query.lower())
    t = re.findall(r"[a-zA-Z0-9']+", text.lower())
    if not q or not t:
        return 0.0
    qset = set(q)
    hits = sum(1 for w in t if w in qset)
    return hits / max(1, len(t))

def retrieve(query: str, chunks: List[Dict[str, Any]], cfg: Config) -> Tuple[List[Dict[str, Any]], float]:
    scored = []
    for c in chunks:
        s = keyword_score(query, c["text"] + " " + c["title"])
        scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)

    top = [c for s, c in scored[:cfg.top_k] if s > 0]
    conf = float(min(1.0, scored[0][0])) if scored else 0.0
    return top, conf


### Tools


In [ ]:
def safe_calculator(expr: str) -> str:
    if len(expr) > 80:
        return "Expression too long."
    if re.search(r"[A-Za-z_]|\.|\[|\]|\{|\}|;|:", expr):
        return "Unsupported expression."
    try:
        val = eval(expr, {"__builtins__": {}}, {})
        if isinstance(val, (int, float)) and (not math.isfinite(val) or abs(val) > 1e12):
            return "Result out of range."
        return str(val)
    except Exception:
        return "Could not evaluate expression."

def tool_rag_search(query: str, cfg: Config) -> Dict[str, Any]:
    hits, conf = retrieve(query, CHUNKS, cfg)
    return {"hits": hits, "confidence": conf}

def tool_calculate(query: str, cfg: Config) -> Dict[str, Any]:
    return {"result": safe_calculator(query)}


### Safety Guardrails


In [ ]:
UNSAFE_TERMS = [
    "how to build a bomb",
    "make a weapon",
]

def guardrail_check_input(user_input: str) -> Optional[str]:
    text = user_input.lower().strip()
    if len(text) > 2000:
        return "I can't process inputs that long in this demo. Please shorten your question."
    for term in UNSAFE_TERMS:
        if term in text:
            return "I can't help with that request."
    return None

def strip_prompt_injection(retrieved_text: str) -> str:
    lines = retrieved_text.splitlines()
    cleaned = []
    for ln in lines:
        if re.search(r"(ignore previous|system prompt|you are chatgpt|developer message)", ln.lower()):
            continue
        cleaned.append(ln)
    return "\n".join(cleaned)

def guardrail_check_output(answer: str) -> Optional[str]:
    # TODO: add domain-specific output rules, e.g. no medical/legal advice.
    return None


### Pipeline Execution


In [ ]:
def build_answer_prompt(user_input: str, retrieved: List[Dict[str, Any]]) -> str:
    sources_block = ""
    if retrieved:
        formatted = []
        for i, c in enumerate(retrieved, start=1):
            snippet = strip_prompt_injection(c["text"])
            formatted.append(f"[{i}] {c['title']}: {snippet}")
        sources_block = "\n\nSOURCES:\n" + "\n\n".join(formatted)

    return (
        "You are a helpful assistant.\n"
        "If you use sources, cite them like [1], [2].\n"
        "If you are unsure, say so and ask a follow-up question.\n\n"
        f"USER QUESTION:\n{user_input}\n"
        f"{sources_block}\n\n"
        "ANSWER:"
    )


In [ ]:
@dataclass
class PipelineResult:
    answer: str
    sources: List[Dict[str, Any]]
    tool_trace: List[str]
    warnings: List[str]
    metadata: Dict[str, Any]

def route(user_input: str, cfg: Config) -> str:
    if not cfg.use_router:
        return "direct"
    if re.search(r"\d", user_input) and any(op in user_input for op in ["+", "-", "*", "/", "**"]):
        return "calculator"
    return "rag" if cfg.use_rag else "direct"

def run_pipeline(user_input: str, cfg: Config) -> PipelineResult:
    tool_trace = []
    warnings = []

    # Input guardrails
    if cfg.guardrails_on:
        refusal = guardrail_check_input(user_input)
        if refusal:
            return PipelineResult(
                answer=refusal,
                sources=[],
                tool_trace=["guardrail->refuse"],
                warnings=[],
                metadata={"ts": now_ts(), "cfg_hash": stable_hash(asdict(cfg))},
            )

    r = route(user_input, cfg)
    tool_trace.append(f"router->{r}")

    retrieved = []
    retrieval_conf = 0.0

    if r == "calculator":
        out = tool_calculate(user_input, cfg)
        answer = f"The result is: {out['result']}"
        tool_trace.append("tool->calculator")

    elif r == "rag":
        out = tool_rag_search(user_input, cfg)
        retrieved = out["hits"]
        retrieval_conf = float(out["confidence"])
        tool_trace.append(f"tool->rag_search (hits={len(retrieved)})")

        if cfg.require_citations and not retrieved:
            warnings.append("No sources retrieved. Answer may be unreliable.")
        if retrieval_conf < cfg.low_confidence_threshold:
            warnings.append("Low retrieval confidence — consider verifying sources or rephrasing.")

        prompt = build_answer_prompt(user_input, retrieved)
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    else:
        prompt = build_answer_prompt(user_input, [])
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    # Output guardrails
    if cfg.guardrails_on:
        out_refusal = guardrail_check_output(answer)
        if out_refusal:
            tool_trace.append("guardrail->output_block")
            answer = out_refusal

    result = PipelineResult(
        answer=answer,
        sources=[{"title": c["title"], "source": c.get("source",""), "snippet": c["text"][:280]} for c in retrieved],
        tool_trace=tool_trace,
        warnings=warnings,
        metadata={
            "ts": now_ts(),
            "route": r,
            "retrieval_conf": retrieval_conf,
            "cfg_hash": stable_hash(asdict(cfg)),
        }
    )

    append_jsonl(cfg.log_path, {
        "ts": result.metadata["ts"],
        "user_input": user_input,
        "answer": result.answer,
        "sources": result.sources,
        "tool_trace": result.tool_trace,
        "warnings": result.warnings,
        "metadata": result.metadata,
        "cfg": asdict(cfg),
    })

    return result


## 5) Launch Gradio App


In [ ]:
import gradio as gr

def gradio_respond(message: str, history: List[Tuple[str, str]]):
    if not message.strip():
        return history, "", "", ""

    res = run_pipeline(message, cfg)
    history = history + [(message, res.answer)]

    sources_md = (
        "\n\n".join([
            f"**[{i+1}] {s['title']}**\n> {s['snippet']}"
            for i, s in enumerate(res.sources)
        ])
        if res.sources else "_No sources retrieved._"
    )
    trace_md  = " → ".join(res.tool_trace) if res.tool_trace else "_(none)_"
    warn_md   = "\n".join([f"⚠️ {w}" for w in res.warnings]) if res.warnings else "✅ No warnings."

    return history, sources_md, trace_md, warn_md

def clear_all():
    return [], "", "", ""

css = """
#title    { text-align: center; padding: 0.5rem 0 0; }
#subtitle { text-align: center; color: #888; font-size: 0.9rem; margin-top: -0.5rem; }
#ask-btn  { background: #e10600 !important; color: white !important; border: none !important; }
"""

EXAMPLES = [
    "What is the penalty for exceeding the pit lane speed limit?",
    "Can a driver change direction more than once to defend their position?",
    "Why was Nico Hulkenberg given no further action in the 2025 Abu Dhabi Grand Prix?",
    "What is DRS and when is a driver allowed to use it?",
    "What penalties apply for an unsafe pit release where no contact occurs?",
]

with gr.Blocks(css=css, title="F1 Rule & Penalty Interpreter") as demo:

    gr.Markdown("# 🏎️ F1 Rule & Penalty Interpreter", elem_id="title")
    gr.Markdown(
        "Ask any question about FIA regulations or steward decisions — "
        "answers are grounded in official documents with citations.",
        elem_id="subtitle"
    )

    with gr.Tabs():

        with gr.Tab("Ask a question"):
            with gr.Row():

                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        label="Conversation",
                        min_height=420,
                        bubble_full_width=False,
                    )
                    with gr.Row():
                        msg = gr.Textbox(
                            placeholder="e.g. What is the penalty for a false start?",
                            show_label=False,
                            scale=5,
                            container=False,
                        )
                        ask_btn = gr.Button("Ask", elem_id="ask-btn", scale=1)
                    clear_btn = gr.Button("🗑 Clear conversation", size="sm", variant="secondary")
                    gr.Examples(examples=EXAMPLES, inputs=msg, label="Example questions")

                with gr.Column(scale=2):
                    gr.Markdown("### 📄 Sources retrieved")
                    sources = gr.Markdown(value="_Sources will appear here after your first question._")
                    with gr.Accordion("🔧 Pipeline trace & warnings", open=False):
                        trace    = gr.Markdown(value="_(none yet)_")
                        warnings = gr.Markdown(value="_(none yet)_")

        with gr.Tab("About this system"):
            gr.Markdown("""
## What this is
The **F1 Rule & Penalty Interpreter** helps F1 fans understand FIA regulations
and steward decisions in plain English.

## How it works
1. **Ingest** — FIA Sporting Regulations and steward decision PDFs are loaded into a knowledge base.
2. **Retrieve** — The system finds the most relevant document chunks using embedding-based similarity search.
3. **Generate** — GPT-4o-mini produces a cited answer grounded in those chunks.
4. **Guardrails** — Inputs are validated, low-confidence answers are flagged, and prompt injection is stripped from retrieved text.

## Limitations
- Coverage is limited to documents in the knowledge base; very recent decisions may be missing.
- Answers should be treated as a starting point, not legal authority.
""")

    submit_args = dict(
        fn=gradio_respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, sources, trace, warnings],
    )
    msg.submit(**submit_args)
    ask_btn.click(**submit_args)
    clear_btn.click(fn=clear_all, outputs=[chatbot, sources, trace, warnings])

demo

/tmp/ipykernel_1580/4159806487.py:20: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat", height=280)
/tmp/ipykernel_1580/4159806487.py:20: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", height=280)


Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7872604970e0>
 |-<gradio.components.chatbot.Chatbot object at 0x78726525df40>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x78726525df40>
 |-<gradio.components.markdown.Markdown object at 0x78729d086000>
 |-<gradio.components.markdown.Markdown object at 0x787292f19bb0>
 |-<gradio.components.markdown.Markdown object at 0x787290e22c90>
fn_index=1
 inputs:
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x78726525df40>
 |-<gradio.components.markdown.Markdown object at 0x78729d086000>
 |-<gradio.components.markdown.Markdown object at 0x787292f19bb0>
 |-<gradio.components.markdown.Markdown object at 0x787290e22c90>

In [ ]:
# --- Launch Gradio App ---
# In Colab, you can set share=True to get a public link (if allowed by your course policy).
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5418c65b24c1f32365.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Some questions to ask

1. **Query:** *"Why was Nico Hulkenbergs collision in the 2025 abu dhabi grand prix given no further action?"*
   - **Ideal Response:** Mentions the decision from the Abu Dhabi Grand Prix for Car 27 and cites the stewards' conclusion. Note: Because of our low chunk size (observed failure mode), it might hallucinate the exact reasoning.

2. **Query:** *"What is the penalty for exceeding the pit lane speed limit?"*
   - **Ideal Response:** Summarizes the regulations related to pit lane speeding and references the decisions applied to Car 23 or Car 18 in Qatar/Abu Dhabi.

3. **Query:** *"Is there a penalty for changing direction more than once to defend a position?"*


## GenAI Ops Experiment

We will run three controlled experiments to evaluate how retrieval design choices affect the quality of answers in the F1 Rule & Penalty Interpreter.

1. Retrieval Method: keyword vs embedding retrieval  
2. Chunk size: small vs medium vs large chunks  
3. Top-k: how many chunks are passed to the model  

Below, we are creating a validation set for these experiments.

In [ ]:
# 15 F1 questions across regulations and steward decisions.
# Each has an "expected" field describing what a correct answer must mention.
# Used across all three experiments so results are comparable.

F1_TESTS = [
    {"id": "q01",
     "question": "What is the penalty for exceeding the pit lane speed limit?",
     "expected": "Time penalty (typically 5 seconds), references pit lane speed limit article"},
    {"id": "q02",
     "question": "What constitutes an unsafe release in the pit lane?",
     "expected": "Releasing a car that endangers another driver or person, references unsafe release regulation"},
    {"id": "q03",
     "question": "Can a driver change direction more than once when defending their position?",
     "expected": "No — only one change of direction is permitted to defend, references defending regulations"},
    {"id": "q04",
     "question": "What is DRS and when is a driver allowed to use it?",
     "expected": "Drag Reduction System, allowed in designated zones when within 1 second of car ahead"},
    {"id": "q05",
     "question": "What happens if a driver causes a collision with another car?",
     "expected": "Typically a time penalty (5 or 10 seconds) or drive-through depending on severity"},
    {"id": "q06",
     "question": "What was the steward decision for Nico Hulkenberg in the 2025 Abu Dhabi Grand Prix?",
     "expected": "No further action, references Abu Dhabi steward decision for Car 27"},
    {"id": "q07",
     "question": "What is the penalty for overtaking another car while the Safety Car is deployed?",
     "expected": "Drive-through penalty or time penalty, references safety car overtaking rules"},
    {"id": "q08",
     "question": "What are the track limits rules and consequences for violations?",
     "expected": "All four wheels must stay within white lines; lap times deleted for violations"},
    {"id": "q09",
     "question": "How do stewards determine if a driver gained a lasting advantage from an incident?",
     "expected": "Stewards assess whether the driver gained a position or time advantage not given back"},
    {"id": "q10",
     "question": "What is the penalty for failing to stop at the weigh bridge when called?",
     "expected": "Exclusion from the race results (DSQ), references weighing regulations"},
    {"id": "q11",
     "question": "Can stewards review and change a penalty decision after the race has finished?",
     "expected": "Yes, within a defined time window if new evidence emerges"},
    {"id": "q12",
     "question": "What penalty applies for a false start?",
     "expected": "Drive-through penalty or stop-and-go penalty, references false start regulations"},
    {"id": "q13",
     "question": "What is the difference between a 5-second and a 10-second time penalty?",
     "expected": "Severity of infringement determines which applies; 5s for minor, 10s for more significant incidents"},
    {"id": "q14",
     "question": "What are the rules about a driver fighting for position on track?",
     "expected": "Must leave racing room, one change of direction to defend, cannot force another car off track"},
    {"id": "q15",
     "question": "What penalties usually apply for an unsafe pit release where no contact occurs?",
     "expected": "Time penalty, references unsafe release decisions and relevant sporting regulation article"},
]

print(f"Test set loaded: {len(F1_TESTS)} questions")

Test set loaded: 15 questions


### Using LLM As A Judge

We are using this to automatically score whether an answer is correct or not. After the pipeline answers each of the 15 test questions, we are using a second LLM call to check whether the the answer is correct or not. This would return a 0 or 1 along with a one sentence reason.

In [ ]:
def llm_judge(question: str, answer: str, expected: str, cfg: Config) -> Dict[str, Any]:
    """Score an answer 0 or 1 against expected criteria using an LLM judge."""
    judge_prompt = (
        "You are an expert evaluator for an F1 rules and regulations assistant.\n\n"
        f"QUESTION: {question}\n\n"
        f"EXPECTED CRITERIA: {expected}\n\n"
        f"ACTUAL ANSWER: {answer}\n\n"
        "Score the actual answer:\n"
        "  1 (CORRECT) — addresses the key points in the expected criteria\n"
        "  0 (INCORRECT) — wrong, hallucinated, or misses key points\n\n"
        'Respond ONLY with a JSON object, no markdown fences: {"score": 0 or 1, "reason": "one sentence"}'
    )
    judge_cfg = Config(**asdict(cfg))
    judge_cfg.use_rag = False
    judge_cfg.guardrails_on = False
    out = call_llm(judge_prompt, judge_cfg)
    try:
        raw = re.sub(r"```json|```", "", out["text"]).strip()
        parsed = json.loads(raw)
        return {"score": int(parsed.get("score", 0)), "reason": parsed.get("reason", "")}
    except Exception as e:
        return {"score": 0, "reason": f"Parse error: {e} | raw: {out['text'][:120]}"}


def run_eval_f1(cfg_eval: Config,
                tests: List[Dict[str, Any]],
                chunks_override: Optional[List[Dict[str, Any]]] = None,
                emb_override=None) -> pd.DataFrame:
    """Run the pipeline on every test question and score with LLM-as-judge."""
    rows = []
    for ex in tests:
        res = run_pipeline(ex["question"], cfg_eval,
                           chunks_override=chunks_override,
                           emb_override=emb_override)
        judgment = llm_judge(ex["question"], res.answer, ex["expected"], cfg_eval)
        rows.append({
            "id": ex["id"],
            "question": ex["question"][:60] + "…",
            "route":    res.metadata.get("route", ""),
            "has_sources":    len(res.sources) > 0,
            "has_citation":   bool(re.search(r"\[\d+\]", res.answer)),
            "retrieval_conf": round(res.metadata.get("retrieval_conf", 0.0), 3),
            "warnings":       len(res.warnings),
            "correct":        judgment["score"],
            "judge_reason":   judgment["reason"],
            "answer_snippet": res.answer[:200],
        })
    return pd.DataFrame(rows)


def summarize_f1(df: pd.DataFrame, label: str = "") -> Dict[str, Any]:
    return {
        "label":            label,
        "n":                len(df),
        "pct_correct":      round(df["correct"].mean() * 100, 1),
        "citation_rate":    round(df["has_citation"].mean() * 100, 1),
        "sources_rate":     round(df["has_sources"].mean() * 100, 1),
        "avg_retrieval_conf": round(df["retrieval_conf"].mean(), 3),
        "avg_warnings":     round(df["warnings"].mean(), 2),
    }

### Embedding Based Retrieval

We are adding the actual embedding retrieval logic here. embed_chunks pre-computes a vector for every chunk in the knowledge base once at startup, and retrieve_embedding finds the most semantically similar chunks to a query using cosine similarity. This is what Experiment 1 is testing against keyword retrieval.

run_pipeline is replaced with a version that accepts chunks_override and emb_override. This is to allow the other experiments to work because Experiment 2 needs to swap different chunk sizes.

In [ ]:
import numpy as np

def cosine_sim(a, b) -> float:
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 1e-10 else 0.0

def clean_text_for_embedding(text: str, max_chars: int = 6000) -> str:
    """
    Sanitize chunk text before sending to the embeddings API:
      - Remove null bytes and other control characters
      - Strip leading/trailing whitespace
      - Truncate to max_chars to stay within token limits
    """
    text = text.replace("\x00", " ")
    text = "".join(c if c.isprintable() or c in "\n\t" else " " for c in text)
    text = text.strip()
    return text[:max_chars]

def embed_chunks(chunks: List[Dict[str, Any]]) -> np.ndarray:
    """Pre-compute a (N, D) embedding matrix for a chunk list."""
    print(f"  Embedding {len(chunks)} chunks …", end=" ", flush=True)
    vecs = []
    failed = 0
    for i, c in enumerate(chunks):
        try:
            cleaned = clean_text_for_embedding(c["text"])
            vec = get_text_embedding(cleaned)
            vecs.append(vec)
        except Exception as e:
            # On failure, use a zero vector so indices stay aligned
            failed += 1
            dim = len(vecs[0]) if vecs else 1536
            vecs.append([0.0] * dim)
        if (i + 1) % 50 == 0:
            print(f"{i+1}", end=" ", flush=True)
    print(f"done. ({failed} failed → zero vector)")
    return np.array(vecs, dtype=float)

def retrieve_embedding(query: str,
                       chunks: List[Dict[str, Any]],
                       chunk_embeddings: np.ndarray,
                       cfg: Config) -> Tuple[List[Dict[str, Any]], float]:
    q_emb = np.array(get_text_embedding(clean_text_for_embedding(query)), dtype=float)
    sims = [cosine_sim(q_emb, ce) for ce in chunk_embeddings]
    scored = sorted(zip(sims, chunks), key=lambda x: x[0], reverse=True)
    top = [c for s, c in scored[:cfg.top_k] if s > 0.15]
    conf = float(scored[0][0]) if scored else 0.0
    return top, conf

# Pre-compute embeddings for the default CHUNKS (used by Exp 1 + Exp 3)
print("Pre-computing embeddings for default CHUNKS …")
CHUNK_EMBEDDINGS = embed_chunks(CHUNKS)
print(f"Embedding matrix shape: {CHUNK_EMBEDDINGS.shape}")

Pre-computing embeddings for default CHUNKS …
  Embedding 17249 chunks … 50 100 150 200 250 300 350 400 450 500 550 600 650 700 750 800 850 900 950 1000 1050 1100 1150 1200 1250 1300 1350 1400 1450 1500 1550 1600 1650 1700 1750 1800 1850 1900 1950 2000 2050 2100 2150 2200 2250 2300 2350 2400 2450 2500 2550 2600 2650 2700 2750 2800 2850 2900 2950 3000 3050 3100 3150 3200 3250 3300 3350 3400 3450 3500 3550 3600 3650 3700 3750 3800 3850 3900 3950 4000 4050 4100 4150 4200 4250 4300 4350 4400 4450 4500 4550 4600 4650 4700 4750 4800 4850 4900 4950 5000 5050 5100 5150 5200 5250 5300 5350 5400 5450 5500 5550 5600 5650 5700 5750 5800 5850 5900 5950 6000 6050 6100 6150 6200 6250 6300 6350 6400 6450 6500 6550 6600 6650 6700 6750 6800 6850 6900 6950 7000 7050 7100 7150 7200 7250 7300 7350 7400 7450 7500 7550 7600 7650 7700 7750 7800 7850 7900 7950 8000 8050 8100 8150 8200 8250 8300 8350 8400 8450 8500 8550 8600 8650 8700 8750 8800 8850 8900 8950 9000 9050 9100 9150 9200 9250 9300 9350 9400 9450 95

Updated run_pipeline():

In [ ]:
def run_pipeline(user_input: str,
                 cfg: Config,
                 chunks_override: Optional[List[Dict[str, Any]]] = None,
                 emb_override=None) -> PipelineResult:
    tool_trace, warnings = [], []

    active_chunks = chunks_override if chunks_override is not None else CHUNKS
    active_embs   = emb_override   if emb_override   is not None else CHUNK_EMBEDDINGS

    if cfg.guardrails_on:
        refusal = guardrail_check_input(user_input)
        if refusal:
            return PipelineResult(
                answer=refusal, sources=[], tool_trace=["guardrail->refuse"],
                warnings=[], metadata={"ts": now_ts(), "cfg_hash": stable_hash(asdict(cfg))})

    r = route(user_input, cfg)
    tool_trace.append(f"router->{r}")
    retrieved, retrieval_conf = [], 0.0

    if r == "calculator":
        out = tool_calculate(user_input, cfg)
        answer = f"The result is: {out['result']}"
        tool_trace.append("tool->calculator")

    elif r == "rag":
        use_emb = getattr(cfg, "use_embedding_retrieval", False)
        if use_emb:
            retrieved, retrieval_conf = retrieve_embedding(
                user_input, active_chunks, active_embs, cfg)
            tool_trace.append(f"tool->embedding_search (hits={len(retrieved)})")
        else:
            out = tool_rag_search(user_input, cfg)
            retrieved, retrieval_conf = out["hits"], float(out["confidence"])
            tool_trace.append(f"tool->keyword_search (hits={len(retrieved)})")

        if cfg.require_citations and not retrieved:
            warnings.append("No sources retrieved. Answer may be unreliable.")
        if retrieval_conf < cfg.low_confidence_threshold:
            warnings.append("Low retrieval confidence — consider rephrasing.")

        prompt = build_answer_prompt(user_input, retrieved)
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]
    else:
        prompt = build_answer_prompt(user_input, [])
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    if cfg.guardrails_on:
        out_refusal = guardrail_check_output(answer)
        if out_refusal:
            tool_trace.append("guardrail->output_block")
            answer = out_refusal

    result = PipelineResult(
        answer=answer,
        sources=[{"title": c["title"], "source": c.get("source", ""),
                  "snippet": c["text"][:280]} for c in retrieved],
        tool_trace=tool_trace, warnings=warnings,
        metadata={"ts": now_ts(), "route": r,
                  "retrieval_conf": retrieval_conf,
                  "cfg_hash": stable_hash(asdict(cfg))})

    append_jsonl(cfg.log_path, {
        "ts": result.metadata["ts"], "user_input": user_input,
        "answer": result.answer, "sources": result.sources,
        "tool_trace": result.tool_trace, "warnings": result.warnings,
        "metadata": result.metadata, "cfg": asdict(cfg)})
    return result


## Experiment 1 - Retrieval Method: Keyword vs Embedding

We currently use keyword matching to retrieve relevant chunks from the FIA documents. Keyword retrieval scores chunks by counting how many query words appear in the chunk text. This experiment tests whether replacing it with embedding-based retrieval, which compares the semantic meaning of the query against chunks using cosine similarity, produces more correct answers.

**Question:** Does embedding-based retrieval improve answer correctness over keyword matching on F1 regulation questions?

**Hypothesis:** Embedding retrieval will outperform keyword matching because it captures semantic similarity rather than exact word overlap.

**What Changes:** use_embedding_retrieval: False (keyword) → True (embedding). This toggle will decide when embeddings are used or not.  

**What is Held Constant:** chunk_size=300, chunk_overlap=50, top_k=4, temperature=0.2, same 15 test questions

**Metric:** % correct answers (LLM is the judge)


In [ ]:
print("EXPERIMENT 1: Keyword vs Embedding Retrieval")

cfg_1a = Config(**asdict(cfg))
cfg_1a.use_embedding_retrieval = False   # A - keyword baseline

cfg_1b = Config(**asdict(cfg))
cfg_1b.use_embedding_retrieval = True    # B - embedding

df_1a = run_eval_f1(cfg_1a, F1_TESTS)
df_1b = run_eval_f1(cfg_1b, F1_TESTS)

sum_1a = summarize_f1(df_1a, "A: Keyword")
sum_1b = summarize_f1(df_1b, "B: Embedding")

print("\nSummary: ")
display(pd.DataFrame([sum_1a, sum_1b]).set_index("label"))
print("\nPer-question delta: ")
cmp1 = (df_1a[["id","question","correct"]].rename(columns={"correct":"A_correct"})
        .merge(df_1b[["id","correct"]].rename(columns={"correct":"B_correct"}), on="id"))
cmp1["delta(B-A)"] = cmp1["B_correct"] - cmp1["A_correct"]
display(cmp1)


EXPERIMENT 1: Keyword vs Embedding Retrieval

Summary: 


,n,pct_correct,citation_rate,sources_rate,avg_retrieval_conf,avg_warnings
label,,,,,,
A: Keyword,15,80.0,60.0,93.3,0.232,0.4
B: Embedding,15,80.0,93.3,93.3,0.590,0.0



Per-question delta: 


,id,question,A_correct,B_correct,delta(B-A)
0,q01,What is the penalty for exceeding the pit lane...,0,1,1
1,q02,What constitutes an unsafe release in the pit ...,1,1,0
2,q03,Can a driver change direction more than once w...,1,1,0
3,q04,What is DRS and when is a driver allowed to us...,1,1,0
4,q05,What happens if a driver causes a collision wi...,1,1,0
5,q06,What was the steward decision for Nico Hulkenb...,0,0,0
6,q07,What is the penalty for overtaking another car...,1,1,0
7,q08,What are the track limits rules and consequenc...,1,1,0
8,q09,How do stewards determine if a driver gained a...,1,1,0
9,q10,What is the penalty for failing to stop at the...,0,0,0


**Results:** Embedding retrieval achieved an 80% accuracy, and this matched the 80% accuracy of keyword retrieval. However, embedding retrieval significantly improved citation rate from 60.0% to 93.3%. Retrieval confidence also increased from 0.232 to 0.590, and warnings dropped from 0.4 to 0.0 on average.

**Conclusion:** The hypothesis was partially confirmed. While embedding retrieval did not improve overall answer correctness, it significantly improved retrieval quality, as we can see through the higher citation rates and confidence scores, and eliminated warnings entirely. This suggests that embeddings are better at finding relevant supporting context even when phrasing differs.

## Experiment 2 — Chunk Size: Small vs Medium vs Large

When we ingest FIA documents, we split them into fixed-size text chunks before retrieval. The chunk size controls how much text surrounds each retrieved passage. This experiment tests three chunk sizes to find the right balance between retrieval precision and contextual completeness.

**Question:** How does chunk size (200, 500, 1000 characters) affect answer correctness and retrieval confidence?

**Hypothesis:** Small chunks (200) will miss context and match only headers, hurting accuracy. Medium chunks (500) will balance precision with context and perform best overall. Large chunks (1000) will capture full reasoning passages but may dilute retrieval relevance for specific questions.

**What Changes:** chunk_size: 200 (small), 500 (medium), 1000 (large)

**What is Held Constant:** keyword retrieval, top_k=4, temperature=0.2, same 15 test questions

**Metric:** % correct answers (LLM is the judge)

In [ ]:
print("EXPERIMENT 2: Chunk Size - Small / Medium / Large")

chunk_configs = [
    ("small",  200,  50),
    ("medium", 500, 125),
    ("large", 1000, 250),
]

exp2_results = {}
for label, csize, coverlap in chunk_configs:
    cfg_x = Config(**asdict(cfg))
    cfg_x.chunk_size    = csize
    cfg_x.chunk_overlap = coverlap
    cfg_x.use_embedding_retrieval = False

    print(f"\nBuilding chunks for size={csize} ", end=" ")
    chunks_x = build_chunks(KB, cfg_x)
    print(f"{len(chunks_x)} chunks")

    print(f"Running eval ({label}) ")
    df_x = run_eval_f1(cfg_x, F1_TESTS, chunks_override=chunks_x)
    exp2_results[label] = (df_x, cfg_x, csize)

print("\nSummary: ")
sum2_rows = []
for label, (df_x, _, csize) in exp2_results.items():
    s = summarize_f1(df_x, f"{label} (size={csize})")
    sum2_rows.append(s)
display(pd.DataFrame(sum2_rows).set_index("label"))

print("\nPer-question delta: ")
cmp2 = exp2_results["small"][0][["id","question"]].copy()
for label, (df_x, _, _) in exp2_results.items():
    cmp2[label] = df_x["correct"].values
display(cmp2)


EXPERIMENT 2: Chunk Size - Small / Medium / Large

Building chunks for size=200  28396 chunks
Running eval (small) 

Building chunks for size=500  11473 chunks
Running eval (medium) 

Building chunks for size=1000  5869 chunks
Running eval (large) 

Summary: 


,n,pct_correct,citation_rate,sources_rate,avg_retrieval_conf,avg_warnings
label,,,,,,
small (size=200),15,80.0,60.0,93.3,0.232,0.4
medium (size=500),15,80.0,60.0,93.3,0.232,0.4
large (size=1000),15,80.0,60.0,93.3,0.232,0.4



Per-question delta: 


,id,question,small,medium,large
0,q01,What is the penalty for exceeding the pit lane...,0,0,0
1,q02,What constitutes an unsafe release in the pit ...,1,1,1
2,q03,Can a driver change direction more than once w...,1,1,1
3,q04,What is DRS and when is a driver allowed to us...,1,1,1
4,q05,What happens if a driver causes a collision wi...,1,1,1
5,q06,What was the steward decision for Nico Hulkenb...,0,0,0
6,q07,What is the penalty for overtaking another car...,1,1,1
7,q08,What are the track limits rules and consequenc...,1,1,1
8,q09,How do stewards determine if a driver gained a...,1,1,1
9,q10,What is the penalty for failing to stop at the...,0,0,0


**Results:** All three chunk sizes (200, 500, and 1000) achieved identical performance across all metrics. Each configuration resulted in 80% accuracy, 60.0% citation rate, 93.3% sources rate, 0.232 average retrieval confidence, and 0.4 average warnings. There was no variation in per-question correctness across chunk sizes, indicating that chunking had no observable impact on system performance for this validation set.

**Conclusion:** The hypothesis was not confirmed. Changing chunk size did not affect retrieval or answer quality in this experiment. This suggests that, for the current dataset and retrieval method, relevant information is contained within relatively small spans of text, and even the smallest chunk size was sufficient to capture the necessary context.

## Experiment 3 — Top-k: How Many Chunks Are Passed to the Model

After retrieval, the top-k most relevant chunks are included in the LLM prompt as context. A low k means less context, the model may miss key information. A high k means more context, but also more noise from less relevant chunks, and a longer, more expensive prompt. This experiment tries to find an ideal spot for our F1 question answering task.

**Question:** How does the number of retrieved chunks passed to the LLM (k=2, k=4, k=8) affect answer correctness and citation rate?

**Hypothesis:** k=2 will be too sparse, missing key context for multi-part questions. k=4 will strike the best balance, enough context without overloading the prompt. k=8 will not significantly improve over k=4 and may hurt by introducing noisy or irrelevant chunks that confuse the LLM.

**What Changes:** top_k: 2, 4, 8

**What is Held Constant:** embedding retrieval, chunk_size=300, temperature=0.2, same 15 test questions

**Metric:** % correct answers (LLM is the judge)

In [ ]:
print("EXPERIMENT 3: Top-k - 2 / 4 / 8")

topk_values = [2, 4, 8]
exp3_results = {}

for k in topk_values:
    cfg_k = Config(**asdict(cfg))
    cfg_k.top_k = k
    cfg_k.use_embedding_retrieval = True   # embedding for a cleaner comparison

    print(f"\nRunning eval (top_k={k}) ")
    df_k = run_eval_f1(cfg_k, F1_TESTS)
    exp3_results[k] = df_k

print("\nSummary: ")
sum3_rows = [summarize_f1(df, f"top_k={k}") for k, df in exp3_results.items()]
display(pd.DataFrame(sum3_rows).set_index("label"))

print("\nPer-question delta: ")
cmp3 = exp3_results[2][["id","question"]].copy()
for k, df in exp3_results.items():
    cmp3[f"k={k}"] = df["correct"].values
display(cmp3)


EXPERIMENT 3: Top-k - 2 / 4 / 8

Running eval (top_k=2) 

Running eval (top_k=4) 

Running eval (top_k=8) 

Summary: 


,n,pct_correct,citation_rate,sources_rate,avg_retrieval_conf,avg_warnings
label,,,,,,
top_k=2,15,80.0,86.7,93.3,0.59,0.0
top_k=4,15,73.3,86.7,93.3,0.59,0.0
top_k=8,15,80.0,93.3,93.3,0.59,0.0



Per-question delta: 


,id,question,k=2,k=4,k=8
0,q01,What is the penalty for exceeding the pit lane...,1,1,1
1,q02,What constitutes an unsafe release in the pit ...,1,1,1
2,q03,Can a driver change direction more than once w...,1,1,1
3,q04,What is DRS and when is a driver allowed to us...,1,1,1
4,q05,What happens if a driver causes a collision wi...,1,1,1
5,q06,What was the steward decision for Nico Hulkenb...,0,0,0
6,q07,What is the penalty for overtaking another car...,1,1,1
7,q08,What are the track limits rules and consequenc...,1,1,1
8,q09,How do stewards determine if a driver gained a...,1,1,1
9,q10,What is the penalty for failing to stop at the...,0,0,0


**Results:** The system achieved 80.0% accuracy at both top_k=2 and top_k=8, while performance dropped to 73.3% at top_k=4. Citation rate improved from 86.7% at top_k=2 and top_k=4 to 93.3% at top_k=8, while sources rate remained constant at 93.3% across all. Retrieval confidence and warnings were unchanged at 0.59 and 0.0 respectively.

**Conclusion:** The hypothesis was partially confirmed. Increasing top_k did not consistently improve accuracy and, in some cases, introduced noise that reduced performance (as seen at top_k=4). While higher k values improved citation coverage, they did not reliably improve answer correctness, suggesting that additional retrieved context can dilute relevance and confuse the model. The best tradeoff appears to be between top_k=2 and top_k=8, with top_k=8 offering better citation coverage without harming overall accuracy.

## Cross-Experiment Summary

In [ ]:
all_rows = []
for s in [sum_1a, sum_1b]:
    all_rows.append({"Experiment": "1 - Retrieval", **{k:v for k,v in s.items() if k!="label"}, "config": s["label"]})
for s in sum2_rows:
    all_rows.append({"Experiment": "2 - Chunk size", **{k:v for k,v in s.items() if k!="label"}, "config": s["label"]})
for s in sum3_rows:
    all_rows.append({"Experiment": "3 - Top-k", **{k:v for k,v in s.items() if k!="label"}, "config": s["label"]})

all_df = pd.DataFrame(all_rows).set_index(["Experiment","config"])
display(all_df[["pct_correct","citation_rate","sources_rate","avg_retrieval_conf"]])


pct_correct  citation_rate  sources_rate  \
Experiment     config                                                        
1 - Retrieval  A: Keyword                80.0           60.0          93.3   
               B: Embedding              80.0           93.3          93.3   
2 - Chunk size small (size=200)          80.0           60.0          93.3   
               medium (size=500)         80.0           60.0          93.3   
               large (size=1000)         80.0           60.0          93.3   
3 - Top-k      top_k=2                   80.0           86.7          93.3   
               top_k=4                   73.3           86.7          93.3   
               top_k=8                   80.0           93.3          93.3   

                                  avg_retrieval_conf  
Experiment     config                                 
1 - Retrieval  A: Keyword                      0.232  
               B: Embedding                    0.590  
2 - Chunk size small (size=200)                0.232  
               medium (size=500)               0.232  
               large (size=1000)               0.232  
3 - Top-k      top_k=2                         0.590  
               top_k=4                         0.590  
               top_k=8                         0.590

## AI Usage

Used Google's gemini within Antigravity to help parse through PDF's and convert these into a format that can be used by the code. Understood how this worked then wrote the code myself.

Used Claude Code to generate a script to scrape the FIA Decision Docs from their website.

Used Claude to help with the LLM as a judge and checking the validation set, updated_pipeline, and embedding, generating only the code for the experiments

For PM3 - We used claude to help us write these experiments. We came up with the ideas on our own, but needed some guidance with implementing them. We referred to claude while creating the code for the exercises as well as the embedding. For the embedding specifically, it was taking a long time to run and we were trying to debug this using claude